# Proyecto final: clasificación de razas de perro por audio

Este notebook distingue cinco razas del dataset DogSpeak usando únicamente herramientas de señales y estadística: normalización, correlación, convolución, transformada rápida de Fourier (FFT), energía y estadísticos descriptivos.

No se usan redes neuronales, embeddings, modelos preentrenados ni librerías de inteligencia artificial. La decisión final se obtiene comparando cada audio con prototipos estadísticos construidos por raza.

Razas: `chihuahua`, `german shepherd`, `husky`, `pitbull` y `shiba inu`. La separación de evaluación se realiza por `dog_id` para evitar que audios del mismo perro aparezcan en entrenamiento y prueba.

## 1. Preparación y configuración

La variable `DATASET_ROOT` debe apuntar a la carpeta que contiene `metadata.csv`. Si los archivos no están disponibles todavía, las celdas de análisis pueden revisarse, pero la extracción y evaluación mostrarán un mensaje explicativo.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from scipy.signal import convolve
from scipy.stats import skew, kurtosis
from IPython.display import Audio, display

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

DATASET_ROOT = Path('DogSpeak')
METADATA_NAME = 'metadata.csv'
MAX_FILES_PER_BREED = 300
MAX_SECONDS = 3.0
TARGET_SAMPLE_RATE = 16000

if not (DATASET_ROOT / METADATA_NAME).exists():
    possible_roots = [Path('/content/DogSpeak'), Path('../DogSpeak'), Path('../../DogSpeak')]
    for possible_root in possible_roots:
        if (possible_root / METADATA_NAME).exists():
            DATASET_ROOT = possible_root
            break

print('Carpeta del dataset:', DATASET_ROOT.resolve())
print('Metadata disponible:', (DATASET_ROOT / METADATA_NAME).exists())

In [ ]:
def load_metadata(dataset_root):
    metadata_path = dataset_root / METADATA_NAME
    if not metadata_path.exists():
        raise FileNotFoundError(
            f'No se encontró {metadata_path}. Descarga DogSpeak y actualiza DATASET_ROOT.'
        )
    metadata = pd.read_csv(metadata_path)
    required_columns = {'filename', 'breed', 'sex', 'dog_id'}
    missing_columns = required_columns.difference(metadata.columns)
    if missing_columns:
        raise ValueError(f'Faltan columnas en metadata.csv: {sorted(missing_columns)}')
    metadata['breed'] = metadata['breed'].astype(str).str.strip().str.lower()
    metadata['dog_id'] = metadata['dog_id'].astype(str)
    return metadata

try:
    metadata = load_metadata(DATASET_ROOT)
    print('Número de registros:', len(metadata))
    print('Razas encontradas:', sorted(metadata['breed'].unique()))
    display(metadata.head())
except (FileNotFoundError, ValueError) as error:
    metadata = None
    print(error)

## 2. Inspección de una vocalización

Primero se observa la señal en el tiempo y su contenido espectral. La convolución con una ventana rectangular produce una envolvente de energía suavizada. La autocorrelación permite estimar periodicidad sin asumir una frecuencia fija.

In [ ]:
def resolve_audio_path(dataset_root, filename):
    relative_path = Path(str(filename))
    candidates = [dataset_root / relative_path, dataset_root / 'dogspeak_released' / relative_path]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

def read_audio(audio_path, max_seconds=MAX_SECONDS, target_sample_rate=TARGET_SAMPLE_RATE):
    signal, sample_rate = sf.read(audio_path, always_2d=False)
    signal = np.asarray(signal, dtype=float)
    if signal.ndim == 2:
        signal = signal.mean(axis=1)
    signal = signal - np.mean(signal)
    if np.max(np.abs(signal), initial=0) > 0:
        signal = signal / np.max(np.abs(signal))
    maximum_samples = int(max_seconds * sample_rate)
    signal = signal[:maximum_samples]
    return signal, sample_rate

if metadata is not None:
    example_row = metadata.iloc[0]
    example_path = resolve_audio_path(DATASET_ROOT, example_row['filename'])
    example_signal, example_rate = read_audio(example_path)
    time_axis = np.arange(len(example_signal)) / example_rate
    smoothing_window = max(3, int(0.025 * example_rate))
    energy_envelope = convolve(example_signal ** 2, np.ones(smoothing_window) / smoothing_window, mode='same')
    spectrum = np.abs(np.fft.rfft(example_signal))
    frequencies = np.fft.rfftfreq(len(example_signal), 1 / example_rate)
    figure, axes = plt.subplots(3, 1, figsize=(12, 9))
    axes[0].plot(time_axis, example_signal, color='steelblue')
    axes[0].set(title=f"Señal: {example_row['breed']} | {example_row['sex']}", xlabel='Tiempo [s]', ylabel='Amplitud')
    axes[1].plot(time_axis, energy_envelope, color='darkorange')
    axes[1].set(title='Energía suavizada mediante convolución', xlabel='Tiempo [s]', ylabel='Energía')
    axes[2].plot(frequencies, spectrum, color='seagreen')
    axes[2].set_xlim(0, min(example_rate / 2, 10000))
    axes[2].set(title='Magnitud de la FFT', xlabel='Frecuencia [Hz]', ylabel='Magnitud')
    plt.tight_layout()
    plt.show()
    display(Audio(example_signal, rate=example_rate))

## 3. Extracción de características con señales y estadística

Las características no son aprendidas por una red. Son cantidades calculadas directamente para cada audio:

- estadísticas temporales: media, desviación, asimetría, curtosis y cruces por cero;
- energía RMS y duración;
- centroide, dispersión y frecuencia dominante del espectro FFT;
- proporción de energía en bandas de frecuencia;
- valor máximo de autocorrelación normalizada fuera del retardo cero;
- variación de la envolvente obtenida mediante convolución.

In [ ]:
FEATURE_NAMES = [
    'duration', 'mean_abs', 'std', 'rms', 'skewness', 'kurtosis', 'zero_crossing_rate',
    'dominant_frequency', 'spectral_centroid', 'spectral_spread', 'spectral_rolloff',
    'band_0_500', 'band_500_1500', 'band_1500_3000', 'band_3000_6000',
    'autocorrelation_peak', 'envelope_variation'
]

def extract_features(signal, sample_rate):
    signal = np.asarray(signal, dtype=float)
    if len(signal) < 32:
        raise ValueError('El audio es demasiado corto para extraer características.')
    absolute_signal = np.abs(signal)
    rms = np.sqrt(np.mean(signal ** 2))
    zero_crossings = np.mean(signal[:-1] * signal[1:] < 0)
    spectrum = np.abs(np.fft.rfft(signal)) ** 2
    frequencies = np.fft.rfftfreq(len(signal), 1 / sample_rate)
    spectrum_sum = np.sum(spectrum) + 1e-12
    spectral_centroid = np.sum(frequencies * spectrum) / spectrum_sum
    spectral_spread = np.sqrt(np.sum((frequencies - spectral_centroid) ** 2 * spectrum) / spectrum_sum)
    cumulative_energy = np.cumsum(spectrum) / spectrum_sum
    rolloff_index = np.searchsorted(cumulative_energy, 0.85)
    rolloff_index = min(rolloff_index, len(frequencies) - 1)
    dominant_frequency = frequencies[np.argmax(spectrum[1:]) + 1] if len(spectrum) > 1 else 0.0
    smoothing_window = max(3, int(0.025 * sample_rate))
    envelope = convolve(signal ** 2, np.ones(smoothing_window) / smoothing_window, mode='same')
    envelope_variation = np.std(envelope) / (np.mean(envelope) + 1e-12)
    autocorrelation = np.correlate(signal, signal, mode='full')[len(signal) - 1:]
    autocorrelation = autocorrelation / (autocorrelation[0] + 1e-12)
    minimum_lag = max(1, int(sample_rate / 1000))
    maximum_lag = min(len(autocorrelation), int(sample_rate / 40))
    autocorrelation_peak = np.max(autocorrelation[minimum_lag:maximum_lag]) if maximum_lag > minimum_lag else 0.0
    bands = [(0, 500), (500, 1500), (1500, 3000), (3000, 6000)]
    band_energies = []
    for lower_frequency, upper_frequency in bands:
        band_mask = (frequencies >= lower_frequency) & (frequencies < upper_frequency)
        band_energies.append(np.sum(spectrum[band_mask]) / spectrum_sum)
    return np.array([
        len(signal) / sample_rate, np.mean(absolute_signal), np.std(signal), rms,
        skew(signal), kurtosis(signal), zero_crossings, dominant_frequency,
        spectral_centroid, spectral_spread, frequencies[rolloff_index], *band_energies,
        autocorrelation_peak, envelope_variation
    ], dtype=float)

def build_feature_table(metadata, dataset_root, max_files_per_breed=MAX_FILES_PER_BREED):
    selected_rows = []
    for breed_name, breed_rows in metadata.groupby('breed'):
        sample_size = min(max_files_per_breed, len(breed_rows))
        selected_rows.append(breed_rows.sample(sample_size, random_state=RANDOM_SEED))
    selected_metadata = pd.concat(selected_rows, ignore_index=True)
    feature_rows = []
    valid_metadata_rows = []
    for row_index, row in selected_metadata.iterrows():
        audio_path = resolve_audio_path(dataset_root, row['filename'])
        try:
            signal, sample_rate = read_audio(audio_path)
            feature_rows.append(extract_features(signal, sample_rate))
            valid_metadata_rows.append(row_index)
        except (OSError, ValueError, RuntimeError) as error:
            print(f'Audio omitido: {audio_path} -> {error}')
    feature_table = pd.DataFrame(
        np.vstack(feature_rows), columns=FEATURE_NAMES, index=valid_metadata_rows
    )
    feature_table = feature_table.join(selected_metadata[['filename', 'breed', 'sex', 'dog_id']])
    return feature_table.reset_index(drop=True)

if metadata is not None:
    feature_table = build_feature_table(metadata, DATASET_ROOT)
    print('Audios procesados:', len(feature_table))
    display(feature_table.groupby('breed')[FEATURE_NAMES[:7]].agg(['mean', 'std']).round(4))

## 4. Comparación de características por raza

Las gráficas sirven para comprobar qué tan separables son las razas antes de clasificar. No se supone que una sola característica resuelva el problema; por eso la decisión usa el vector completo normalizado.

In [ ]:
if metadata is not None and 'feature_table' in locals():
    features_to_plot = ['rms', 'zero_crossing_rate', 'dominant_frequency', 'spectral_centroid', 'autocorrelation_peak']
    figure, axes = plt.subplots(1, len(features_to_plot), figsize=(18, 4))
    for axis, feature_name in zip(axes, features_to_plot):
        feature_table.boxplot(column=feature_name, by='breed', ax=axis, rot=35, grid=False)
        axis.set_title(feature_name)
        axis.set_xlabel('')
    figure.suptitle('Distribución de características por raza')
    plt.tight_layout()
    plt.show()

## 5. Clasificador estadístico por prototipos

Para cada partición se calcula la media y desviación estándar solamente con el conjunto de entrenamiento. Cada raza se representa por la mediana de sus vectores de características. Un audio se asigna a la raza con menor distancia euclidiana en el espacio estandarizado.

La división es 75/25 por perro. Por tanto, la evaluación mide generalización a perros que no participaron en el prototipo.

In [ ]:
def split_by_dog(feature_table, test_fraction=0.25):
    dog_ids = feature_table['dog_id'].drop_duplicates().to_numpy()
    shuffled_dogs = rng.permutation(dog_ids)
    test_count = max(1, int(round(test_fraction * len(shuffled_dogs))))
    test_dogs = set(shuffled_dogs[:test_count])
    test_mask = feature_table['dog_id'].isin(test_dogs)
    return feature_table.loc[~test_mask].copy(), feature_table.loc[test_mask].copy()

def fit_prototype_classifier(training_table):
    training_values = training_table[FEATURE_NAMES].to_numpy(dtype=float)
    location = np.nanmedian(training_values, axis=0)
    scale = np.nanstd(training_values, axis=0)
    scale[scale < 1e-12] = 1.0
    normalized_values = (training_values - location) / scale
    prototypes = {}
    breed_values = training_table['breed'].to_numpy()
    for breed_name in sorted(training_table['breed'].unique()):
        breed_mask = breed_values == breed_name
        prototypes[breed_name] = np.nanmedian(normalized_values[breed_mask], axis=0)
    return {'location': location, 'scale': scale, 'prototypes': prototypes}

def predict_prototypes(model, table):
    values = table[FEATURE_NAMES].to_numpy(dtype=float)
    normalized_values = (values - model['location']) / model['scale']
    breed_names = list(model['prototypes'])
    prototype_matrix = np.vstack([model['prototypes'][breed] for breed in breed_names])
    distances = np.linalg.norm(normalized_values[:, None, :] - prototype_matrix[None, :, :], axis=2)
    predictions = np.array([breed_names[index] for index in np.argmin(distances, axis=1)])
    confidence = 1.0 / (1.0 + np.min(distances, axis=1))
    return predictions, confidence, distances

if metadata is not None and 'feature_table' in locals() and len(feature_table['dog_id'].unique()) >= 2:
    training_table, test_table = split_by_dog(feature_table)
    prototype_model = fit_prototype_classifier(training_table)
    predictions, confidence, distances = predict_prototypes(prototype_model, test_table)
    test_table = test_table.copy()
    test_table['prediction'] = predictions
    test_table['confidence'] = confidence
    accuracy = np.mean(test_table['breed'].to_numpy() == predictions)
    print('Perros en entrenamiento:', training_table['dog_id'].nunique())
    print('Perros en prueba:', test_table['dog_id'].nunique())
    print(f'Exactitud de prueba: {accuracy:.3f}')
    display(pd.crosstab(test_table['breed'], test_table['prediction'], margins=True))

## 6. Matriz de confusión y análisis de errores

La matriz de confusión permite identificar qué pares de razas producen características acústicas parecidas. La confianza mostrada no es una probabilidad: es solo una transformación de la distancia al prototipo y debe interpretarse como un indicador relativo.

In [ ]:
if metadata is not None and 'test_table' in locals():
    breed_order = sorted(feature_table['breed'].unique())
    confusion = pd.crosstab(test_table['breed'], test_table['prediction']).reindex(
        index=breed_order, columns=breed_order, fill_value=0
    )
    figure, axis = plt.subplots(figsize=(7, 6))
    image = axis.imshow(confusion.to_numpy(), cmap='Blues')
    axis.set_xticks(range(len(breed_order)), breed_order, rotation=35, ha='right')
    axis.set_yticks(range(len(breed_order)), breed_order)
    axis.set_xlabel('Raza predicha')
    axis.set_ylabel('Raza real')
    axis.set_title('Matriz de confusión')
    for row_index in range(len(breed_order)):
        for column_index in range(len(breed_order)):
            axis.text(column_index, row_index, confusion.iloc[row_index, column_index], ha='center', va='center')
    figure.colorbar(image, ax=axis)
    plt.tight_layout()
    plt.show()
    display(test_table.loc[test_table['breed'] != test_table['prediction'], ['breed', 'prediction', 'dog_id', 'confidence']].head(20))

## 7. Clasificar un archivo nuevo

Cambie `NEW_AUDIO_PATH` por un WAV externo. El audio se procesa exactamente con las mismas operaciones y se compara con los prototipos ya calculados; no se recalculan los prototipos con el archivo que se quiere clasificar.

In [ ]:
NEW_AUDIO_PATH = Path('audio_nuevo.wav')

if metadata is not None and 'prototype_model' in locals() and NEW_AUDIO_PATH.exists():
    new_signal, new_rate = read_audio(NEW_AUDIO_PATH)
    new_features = extract_features(new_signal, new_rate)
    new_table = pd.DataFrame([new_features], columns=FEATURE_NAMES)
    new_prediction, new_confidence, new_distances = predict_prototypes(prototype_model, new_table)
    print('Raza estimada:', new_prediction[0])
    print(f'Indicador relativo de confianza: {new_confidence[0]:.3f}')
    display(pd.DataFrame({'raza': list(prototype_model['prototypes']), 'distancia': new_distances[0]}).sort_values('distancia'))
elif metadata is not None:
    print(f'No se encontró {NEW_AUDIO_PATH}. Actualiza NEW_AUDIO_PATH para clasificar un audio.')

## Conclusiones para la sustentación

1. La FFT transforma cada vocalización al dominio de frecuencia, mientras que la convolución resume cambios locales de energía.
2. La autocorrelación aporta información de periodicidad y complementa las características espectrales.
3. La clasificación no depende de IA: usa prototipos construidos con medianas y una distancia euclidiana después de estandarizar las variables.
4. La evaluación por `dog_id` es necesaria porque varias secuencias del dataset pertenecen al mismo perro.
5. Un resultado bajo no implica que el código esté fallando: puede indicar que raza y vocalización no son suficientemente separables con estas características o que hace falta aumentar la muestra y mejorar el preprocesamiento.